# ANN vs Decision Tree: Cooling Load Prediction
**Dataset**: Energy Efficiency — UCI Machine Learning Repository (ENB2012_data.xlsx)
**Target**: Y2 (Cooling Load)
**Models**: Multi-Layer Perceptron (Keras) vs Decision Tree Regressor (scikit-learn)

## Section 0 — Install Dependencies
Run this cell once to ensure all required libraries are available in your environment.

In [ ]:
# Run only if libraries are not yet installed
%pip install pandas numpy scikit-learn tensorflow openpyxl

## Section 1 — Import Libraries
All required libraries are imported here.
- `pandas` and `numpy`: data manipulation and numerical operations
- `sklearn`: data splitting, preprocessing, Decision Tree model, and evaluation metrics
- `tensorflow.keras`: building the Neural Network (MLP) model
- `time`: measuring training and evaluation runtime for each model

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

## Section 2 — Load and Explore Dataset
The dataset is read from `ENB2012_data.xlsx`.
- Columns X1 to X8 are input features representing building characteristics
- Column Y2 is the prediction target: Cooling Load
- We check the number of rows/columns and whether any missing values exist

In [ ]:
# Read dataset from Excel file
df = pd.read_excel('ENB2012_data.xlsx')

# Display first 5 rows to verify data is loaded correctly
print('First 5 rows:')
print(df.head())

# Check dataset dimensions
print(f'\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns')

# Check for missing values in each column
print('\nMissing values per column:')
print(df.isnull().sum())

# Descriptive statistics
print('\nDescriptive statistics:')
print(df.describe())

## Section 3 — Separate Features and Target
- `X`: all 8 input feature columns (X1 to X8)
- `y`: only column Y2, which is the Cooling Load
- Column Y1 (Heating Load) is excluded since this experiment focuses on Cooling Load only

In [ ]:
# Select 8 input feature columns
feature_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
X = df[feature_cols].values

# Select Y2 as the target (Cooling Load)
y = df['Y2'].values

print(f'X shape (features): {X.shape}')
print(f'y shape (target)  : {y.shape}')

## Section 4 — Train-Test Split and Feature Scaling
Data is split into 80% training and 20% testing.
- `random_state=42` ensures the split is reproducible across runs
- `StandardScaler` normalizes features to zero mean and unit variance
- The scaler is **fitted only on training data**, then applied to both train and test sets
- This avoids data leakage: the test set must not influence the scaling parameters

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data only, then transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## Section 5 — Decision Tree Regressor (Baseline)
Decision Tree Regressor is used as a simple, interpretable baseline model.
- `max_depth=10` limits tree depth to prevent overfitting
- `random_state=42` ensures reproducibility
- Runtime is measured from the start of training to the end of test prediction
- Evaluation metrics: MSE, MAE, and R2 Score

In [ ]:
# Initialize Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)

# Start timer before training
dt_start = time.time()

# Train the model on scaled training data
dt_model.fit(X_train_scaled, y_train)

# Predict on test data
dt_pred = dt_model.predict(X_test_scaled)

# Stop timer after prediction
dt_end = time.time()
dt_runtime = dt_end - dt_start

# Compute evaluation metrics
dt_mse = mean_squared_error(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_r2  = r2_score(y_test, dt_pred)

print('=== Decision Tree Regressor ===')
print(f'MSE    : {dt_mse:.4f}')
print(f'MAE    : {dt_mae:.4f}')
print(f'R2     : {dt_r2:.4f}')
print(f'Runtime: {dt_runtime:.4f} seconds')

## Section 6 — Neural Network MLP (Keras)
An MLP model is built using Keras Sequential API.
- **Input layer**: automatically infers shape from the 8 input features
- **Hidden layer 1**: 64 neurons, ReLU activation — captures non-linear relationships
- **Hidden layer 2**: 32 neurons, ReLU activation — further feature abstraction
- **Output layer**: 1 neuron, no activation — suitable for continuous regression output
- `optimizer='adam'`: adaptive learning rate optimizer, well-suited for regression tasks
- `loss='mse'`: Mean Squared Error as the training loss function
- `EarlyStopping`: halts training if validation loss does not improve for 10 consecutive epochs, preventing overfitting
- Runtime is measured from start of training to end of test prediction

In [ ]:
# Build MLP model architecture
nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),  # Hidden layer 1
    Dense(32, activation='relu'),                                           # Hidden layer 2
    Dense(1)                                                                # Output layer (regression)
])

# Compile model: define optimizer and loss function
nn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# EarlyStopping: stop training if val_loss does not improve for 10 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Start timer before training
nn_start = time.time()

# Train the model
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=200,           # maximum 200 epochs
    batch_size=32,        # 32 samples per gradient update
    validation_split=0.1, # 10% of training data used for validation during training
    callbacks=[early_stop],
    verbose=0             # suppress per-epoch log output
)

# Predict on test data
nn_pred = nn_model.predict(X_test_scaled).flatten()

# Stop timer after prediction
nn_end = time.time()
nn_runtime = nn_end - nn_start

# Compute evaluation metrics
nn_mse = mean_squared_error(y_test, nn_pred)
nn_mae = mean_absolute_error(y_test, nn_pred)
nn_r2  = r2_score(y_test, nn_pred)

print('=== Neural Network MLP (Keras) ===')
print(f'MSE    : {nn_mse:.4f}')
print(f'MAE    : {nn_mae:.4f}')
print(f'R2     : {nn_r2:.4f}')
print(f'Runtime: {nn_runtime:.4f} seconds')
print(f'Epochs completed: {len(history.history["loss"])}')

## Section 7 — Model Comparison
All evaluation metrics from both models are combined into a single comparison table.
This makes it easy to determine which model performs better in terms of accuracy and speed.

In [ ]:
# Build comparison table
results = pd.DataFrame({
    'Model'      : ['Decision Tree', 'Neural Network (MLP)'],
    'MSE'        : [round(dt_mse, 4), round(nn_mse, 4)],
    'MAE'        : [round(dt_mae, 4), round(nn_mae, 4)],
    'R2'         : [round(dt_r2, 4),  round(nn_r2, 4)],
    'Runtime (s)': [round(dt_runtime, 4), round(nn_runtime, 4)]
})

print('=== Model Performance Comparison ===')
print(results.to_string(index=False))

## Section 8 — Analysis and Interpretation
Fill in your analysis here based on the actual results from Section 7.

Points to address:
- Which model has lower MSE and MAE?
- Which model has a higher R2 score (closer to 1.0 = better fit)?
- How large is the runtime difference between Decision Tree and Neural Network?
- Is the added complexity of NN justified by the performance gain over Decision Tree?
- Why is NN slower? (backpropagation across many epochs vs a single tree-building pass)

Example interpretation (adjust with your actual numbers):
'The Neural Network achieved a lower MSE than Decision Tree, indicating more accurate predictions. However, its runtime was significantly longer due to iterative weight optimization across multiple epochs. For a relatively small dataset like this (768 samples), Decision Tree already delivers competitive performance with much faster computation time.'